# 04 — MLflow tracking and registry (Phase 7)

Logs every training run to MLflow (parameters, metrics, slice metrics,
feature list, dataset manifest, model signature, input example, and the
serialized model itself), registers passing candidates as versions of
`rivercast-kaub-6h` / `rivercast-kaub-12h`, and runs the champion/challenger
promotion transaction. All MLflow calls live in `rivercast.models.tracking`
and `rivercast.models.registry`; this notebook only calls them and displays
results (CLAUDE.md rule 17). The same workflow is available from the
terminal:

```bash
rivercast train --horizon 6 --model ridge --promote
```

> RiverCast is **educational**. Water level is relative to the local gauge
> zero — not river depth — and these forecasts must never inform real-world
> decisions.

In [ ]:
from pathlib import Path

from mlflow.client import MlflowClient

from rivercast.config import load_config
from rivercast.envcheck import find_lab_root
from rivercast.models import get_champion, resolve_tracking_uri, train_track_and_register

LAB_ROOT = find_lab_root(Path.cwd())
config = load_config(LAB_ROOT / "configs" / "local.yaml")
FIXTURE_DIR = LAB_ROOT / "data_fixtures" / "pegelonline"
MODELS_DIR = LAB_ROOT / "models" / "local"

tracking_uri = resolve_tracking_uri(config, LAB_ROOT)
print(f"mlflow tracking uri: {tracking_uri}")
print(f"experiment: {config.mlflow.experiment}")
print(f"registered models: {config.mlflow.registered_models}")

## Tracking URI

Fixture-mode workbenches and CI have no live MLflow server, so
`mlflow.tracking_uri_default` in `configs/base.yaml` falls back to a local
sqlite store under `storage.root` (`resolve_tracking_uri`). Setting the
`MLFLOW_TRACKING_URI` environment variable — as an OpenShift AI workbench
data connection would — always takes priority over this default.

## Train, track, and register a candidate for each horizon

`train_track_and_register` composes the full Phase 6 training flow with
Phase 7's tracking and registry: train → log the run to MLflow → register
the candidate as a new model version → evaluate the promotion gates from
`configs/base.yaml` (`thresholds.promotion`) against the current champion's
*real* logged metrics (not a placeholder) → optionally run the promotion
transaction. `promote=True` only moves the `champion` alias if the gates
pass **and** the injected deploy/smoke-test step succeeds — the default stub
always succeeds since real deployment arrives in Phases 8-11, but a failing
stub is exercised further down to prove the champion never moves on a failed
validation.

In [ ]:
outcomes = {}
for horizon in config.horizons_hours:
    model_name = "ridge"  # the Phase 6 report showed ridge beats persistence at both horizons
    outcome = train_track_and_register(
        config, LAB_ROOT, FIXTURE_DIR, horizon, model_name, MODELS_DIR, seed=42, promote=True
    )
    outcomes[horizon] = outcome
    print(
        f"h={horizon:>2}h  {model_name:<8} "
        f"run={outcome.logged_run.run_id[:8]}  "
        f"registered={outcome.registered_model_name} v{outcome.model_version.version}  "
        f"gates={'PASS' if outcome.decision.approved else 'REJECTED'}  "
        f"promoted={outcome.promoted}"
    )

## A rejected candidate stays traceable, never becomes champion

`hist-gradient-boosting` at the 12h horizon underperforms persistence on
this small fixture window (Phase 6 finding, `reports/baseline/baseline_report.md`)
— a real rejection, not a contrived one. It is still registered (so it
remains traceable, per the acceptance criteria) but the promotion gates
reject it and it never receives the `champion` alias.

In [ ]:
rejected_outcome = train_track_and_register(
    config,
    LAB_ROOT,
    FIXTURE_DIR,
    12,
    "hist-gradient-boosting",
    MODELS_DIR,
    seed=42,
    promote=True,
)
print(f"gates: {'PASS' if rejected_outcome.decision.approved else 'REJECTED'}")
print(f"reasons: {rejected_outcome.decision.reasons}")
print(f"promoted: {rejected_outcome.promoted}")
assert not rejected_outcome.decision.approved
assert not rejected_outcome.promoted

client = MlflowClient(tracking_uri=tracking_uri)
fetched = client.get_model_version(
    rejected_outcome.registered_model_name, rejected_outcome.model_version.version
)
print(f"validation_status tag: {fetched.tags['validation_status']}")
assert fetched.tags["validation_status"] == "rejected"

## Champion lookup by alias

Once promoted, the champion for each horizon is loadable purely by alias —
no version number needs to be known ahead of time. This is what the future
serving layer (Phases 8-11) will call.

In [ ]:
for horizon in config.horizons_hours:
    registered_model_name = config.mlflow.registered_models[str(horizon)]
    champion = get_champion(client, registered_model_name)
    if champion is None:
        print(f"h={horizon:>2}h  {registered_model_name}: no champion yet")
    else:
        print(
            f"h={horizon:>2}h  {registered_model_name}: champion is v{champion.version} "
            f"(run {champion.run_id[:8]}, dataset {champion.tags.get('dataset_id', '?')[:19]})"
        )

## A failed deployment validation must not move the champion

The promotion transaction only moves `champion` *after* a successful
deploy-and-smoke-test step (CLAUDE.md rule 14). Re-training the same 6h
model with a deploy/smoke-test stub that always fails proves the existing
champion is left untouched even though the offline promotion gates pass.

In [ ]:
champion_before = get_champion(client, "rivercast-kaub-6h")
print(f"champion before: v{champion_before.version}")

failed_outcome = train_track_and_register(
    config,
    LAB_ROOT,
    FIXTURE_DIR,
    6,
    "ridge",
    MODELS_DIR,
    seed=42,
    promote=True,
    deploy_and_smoke_test=lambda _model_version: False,
)
print(f"gates: {'PASS' if failed_outcome.decision.approved else 'REJECTED'}")
print(f"promoted: {failed_outcome.promoted}")
assert failed_outcome.decision.approved  # offline gates passed
assert not failed_outcome.promoted  # but deployment validation failed

champion_after = get_champion(client, "rivercast-kaub-6h")
print(f"champion after: v{champion_after.version}")
assert champion_after.version == champion_before.version, "champion must not move on deploy failure"

## Conclusion

Every training run — approved or rejected — is now logged to MLflow with
full lineage (parameters, metrics, slices, dataset manifest, feature list,
model signature, Git commit, station UUIDs) and registered as a traceable
model version. Promotion only moves the `champion` alias through the full
transaction: register → challenger → validate → deploy → smoke test → move
champion, and a deployment failure never touches the existing champion.
Next: Phase 8 containerizes the reusable components (fetch, train, evaluate,
register, promote, deploy) that these functions back.